In [49]:
import json
from collections import defaultdict

import gc_utils
import h5py
import numpy as np

In [50]:
sim = "m12m"
# sim_dir = "/Users/z5114326/Documents/simulations/"
sim_dir = "/Volumes/Expansion/simulations/"

proc_file = sim_dir + sim + "/" + sim + "_processed.hdf5"
proc_data = h5py.File(proc_file, "r")  # open processed data file

In [51]:
min_gc_num = 5

snap = 600
snap_id = gc_utils.snapshot_name(snap)

In [52]:
group_counts = defaultdict(list)

# Loop through iterations
for it_id in proc_data.keys():
    snap_data = proc_data[it_id]["snapshots"][snap_id]
    groups = np.abs(snap_data["group_id"][()])

    unique_groups, counts = np.unique(groups, return_counts=True)

    # Store counts *per iteration*
    for g, c in zip(unique_groups, counts):
        group_counts[g].append(c)

# Now compute averages
common_groups = [g for g, cnts in group_counts.items() if np.mean(cnts) >= min_gc_num]

In [53]:
common_groups

[0, 30145433, 37002741, 40978260, 63550494, 76444787]

In [54]:
halt = gc_utils.get_halo_tree(sim, sim_dir)

Retrieving Halo Tree.....................: 100%|████████████████████████████████████████████████████████████████████████| 1/1 [01:15<00:00, 75.12s/it]


In [55]:
snap_order = []
for grp in common_groups:
    if grp == 0:
        snap_order.append(0)

    else:
        halt_idx = np.where(halt["tid"] == grp)[0][0]
        snap_order.append(halt["snapshot"][halt_idx])

sorted_groups = [grp for _, grp in sorted(zip(snap_order, common_groups))]

In [56]:
common_colors = [
    "red",
    "blue",
    "green",
    "cyan",
    "magenta",
    "orange",
    "purple",
    "pink",
    "brown",
    "gold",
    "darkgreen",
    "navy",
    "olive",
    "teal",
    "maroon",
]

In [57]:
group_dict = {sim: {}}

for i, grp in enumerate(sorted_groups):
    group_dict[sim][str(grp)] = common_colors[i]

In [58]:
group_dict[sim]

{'0': 'red',
 '30145433': 'blue',
 '37002741': 'green',
 '40978260': 'cyan',
 '63550494': 'magenta',
 '76444787': 'orange'}

In [59]:
# Save the dictionary to a JSON file
save_file = sim_dir + sim + "/" + "gc_groups.json"
with open(save_file, "w") as json_file:
    json.dump(group_dict, json_file, indent=4)

In [60]:
proc_data.close()